# Notebook 8: ESG & CSRD Compliance Reporting
## *Air quality data for sustainability disclosures and regulatory compliance*

**Target audience:** ESG analysts, sustainability managers, CSRD compliance officers, real estate fund managers, corporate environmental teams

**Regulatory context:**
- **CSRD** (Corporate Sustainability Reporting Directive) — in force Jan 2024, requires large EU companies to disclose environmental impacts including air quality at operated sites
- **ESRS E2** (Environmental Standard) — specifically covers pollution: disclose pollutant levels at material locations
- **EU AQD 2024 revision** — new annual NO₂ limit of 20 µg/m³ from 2030 (currently 40 µg/m³)
- **WHO 2021 guidelines** — NO₂ 10 µg/m³, PM2.5 5 µg/m³ (increasingly used for disclosure)

### What we build
1. **Site compliance screening** — assess each office/facility against WHO + EU AQD standards
2. **WHO exceedance hours** — count hours per year above health guidelines (required for ESRS E2)
3. **Portfolio trend report** — multi-year trajectory for ESG progress reporting
4. **CSRD disclosure table** — ready-to-use table for annual sustainability report

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from jiskta import JisktaClient

API_KEY = os.environ.get('JISKTA_API_KEY', 'your_api_key_here')
client = JisktaClient(API_KEY)

# Regulatory reference values
WHO_ANNUAL = {'no2': 10.0, 'pm2p5':  5.0, 'pm10': 15.0}  # µg/m³ annual mean
EU_AQD     = {'no2': 40.0, 'pm2p5': 25.0, 'pm10': 40.0}  # current EU AQD limits
EU_AQD2030 = {'no2': 20.0, 'pm2p5': 10.0, 'pm10': 20.0}  # 2024 revision (in force 2030)

# WHO 2021 hourly NO2 guideline for exceedance calculation
WHO_HOURLY_NO2 = 25.0   # µg/m³ 1-hour mean (WHO 2021)

print('Ready. Regulatory limits loaded.')

## 1. Office Portfolio Compliance Screening

Screen all company sites against WHO 2021 and EU AQD limits in one batch.
Each site costs 1 credit per pollutant per year queried.

In [ ]:
# Example corporate real estate portfolio (European offices)
PORTFOLIO = [
    {'name': 'Paris HQ',           'lat': 48.877, 'lon': 2.329, 'employees': 450},
    {'name': 'Amsterdam Office',   'lat': 52.370, 'lon': 4.895, 'employees': 120},
    {'name': 'Berlin Office',      'lat': 52.520, 'lon': 13.405, 'employees': 85},
    {'name': 'Munich Office',      'lat': 48.137, 'lon': 11.576, 'employees': 60},
    {'name': 'Lyon Office',        'lat': 45.757, 'lon': 4.832, 'employees': 40},
    {'name': 'Stockholm Office',   'lat': 59.334, 'lon': 18.065, 'employees': 35},
    {'name': 'Warsaw Office',      'lat': 52.229, 'lon': 21.012, 'employees': 55},
    {'name': 'Madrid Office',      'lat': 40.416, 'lon': -3.703, 'employees': 70},
]

YEAR = 2023
VARIABLES = ['no2', 'pm2p5']

screening = []
for site in PORTFOLIO:
    df = client.query(
        lat=site['lat'], lon=site['lon'],
        start=f'{YEAR}-01', end=f'{YEAR}-12',
        variables=VARIABLES,
        aggregate='monthly',
    )
    row = {'Site': site['name'], 'Employees': site['employees']}
    for var in VARIABLES:
        col = f'{var}_mean'
        if col in df.columns:
            ann = df[col].mean()
            row[f'{var.upper()} µg/m³'] = round(ann, 2)
            row[f'{var.upper()} vs WHO'] = f'{ann/WHO_ANNUAL[var]:.1f}×'
            row[f'{var.upper()} EU 2030'] = '✅' if ann <= EU_AQD2030[var] else '⚠️'
    screening.append(row)
    print(f'  {site["name"]:<22} NO₂={row.get("NO2 µg/m³","?"):.2f} µg/m³ '
          f'({row.get("NO2 vs WHO","?")} WHO)  PM2.5={row.get("PM2P5 µg/m³","?"):.2f}')

df_screen = pd.DataFrame(screening)
print(f'\nTotal cost: {len(PORTFOLIO) * len(VARIABLES)} credits')

In [ ]:
# Colour-code compliance status
def style_compliance(val):
    if isinstance(val, str):
        if '×' in val:
            mult = float(val.replace('×', ''))
            if mult <= 1.0: return 'background-color: #dcfce7'
            if mult <= 2.0: return 'background-color: #fef9c3'
            if mult <= 4.0: return 'background-color: #fee2e2'
            return 'background-color: #b91c1c; color: white'
    return ''

display(df_screen.style.applymap(style_compliance))

## 2. WHO Hourly Exceedance Hours (ESRS E2 Metric)

ESRS E2 asks companies to disclose how often pollutant levels exceeded health guidelines.
We use the `exceedance` aggregate to count hours above the WHO hourly NO₂ guideline (25 µg/m³).

In [ ]:
exceedance_results = []

for site in PORTFOLIO:
    exc = client.query(
        lat=site['lat'], lon=site['lon'],
        start=f'{YEAR}-01', end=f'{YEAR}-12',
        variables=['no2'],
        aggregate='exceedance',
        threshold=WHO_HOURLY_NO2,
    )
    total_hours = exc['total_hours'].iloc[0]
    exc_hours   = exc['hours_above'].iloc[0]
    pct_above   = exc['pct_above'].iloc[0]

    exceedance_results.append({
        'Site': site['name'],
        'Employees': site['employees'],
        'Hours above WHO 25 µg/m³': int(exc_hours),
        'Total hours': int(total_hours),
        '% of year': round(pct_above, 1),
        'ESRS E2 status': '✅ Compliant' if exc_hours == 0 else f'⚠️ {int(exc_hours)}h exceeded',
    })
    print(f'  {site["name"]:<22} {int(exc_hours):>5} hours above WHO limit ({pct_above:.1f}% of year)')

df_exc = pd.DataFrame(exceedance_results)
print(f'\nTotal cost: {len(PORTFOLIO)} credits')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
df_exc_sorted = df_exc.sort_values('Hours above WHO 25 µg/m³', ascending=True)

colors = ['#DC2626' if h > 500 else '#EA580C' if h > 100 else '#65A30D'
          for h in df_exc_sorted['Hours above WHO 25 µg/m³']]

bars = ax.barh(df_exc_sorted['Site'], df_exc_sorted['Hours above WHO 25 µg/m³'],
               color=colors, edgecolor='white')

for bar, hours, pct in zip(bars,
                            df_exc_sorted['Hours above WHO 25 µg/m³'],
                            df_exc_sorted['% of year']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{int(hours)}h ({pct:.1f}%)', va='center', fontsize=9)

ax.set_xlabel(f'Hours per year above WHO hourly NO₂ guideline ({WHO_HOURLY_NO2} µg/m³)')
ax.set_title(f'ESRS E2 — WHO Hourly NO₂ Exceedance by Site ({YEAR})\n'
             'Red = >500h (>5.7%), Orange = >100h (>1.1%), Green = <100h', fontsize=12)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.savefig('../exports/08_esrs_exceedance.png', dpi=150, bbox_inches='tight')
plt.show()
display(df_exc[['Site', 'Hours above WHO 25 µg/m³', '% of year', 'ESRS E2 status']])

## 3. Portfolio Trend 2018–2023 — ESG Progress Reporting

Demonstrate year-over-year improvement across the portfolio — a key metric for ESG ratings.

In [ ]:
# Focus on 3 key sites for trend analysis (one API call per site/year)
KEY_SITES = [
    {'name': 'Paris HQ',        'lat': 48.877, 'lon': 2.329},
    {'name': 'Warsaw Office',   'lat': 52.229, 'lon': 21.012},
    {'name': 'Stockholm Office','lat': 59.334, 'lon': 18.065},
]
TREND_YEARS = list(range(2018, 2024))

trend_data = {s['name']: [] for s in KEY_SITES}

for year in TREND_YEARS:
    for site in KEY_SITES:
        df_y = client.query(
            lat=site['lat'], lon=site['lon'],
            start=f'{year}-01', end=f'{year}-12',
            variables=['no2'], aggregate='monthly',
        )
        trend_data[site['name']].append(df_y['no2_mean'].mean())
    print(f'{year}: {" | ".join(f"{s[\"name\"].split()[0]}: {trend_data[s[\"name\"]][-1]:.2f}" for s in KEY_SITES)}')

print(f'\nTotal cost: {len(TREND_YEARS) * len(KEY_SITES)} credits')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
site_colors = ['#2563EB', '#DC2626', '#16A34A']

for site, color in zip(KEY_SITES, site_colors):
    vals = trend_data[site['name']]
    ax.plot(TREND_YEARS, vals, 'o-', label=site['name'], color=color,
            linewidth=2.5, markersize=8)
    # Improvement annotation
    improvement = (vals[-1] - vals[0]) / vals[0] * 100
    ax.text(TREND_YEARS[-1] + 0.1, vals[-1],
            f'{improvement:+.1f}%', fontsize=8, color=color, va='center')

for limit, label, color, ls in [
    (WHO_ANNUAL['no2'],  f'WHO 2021 ({WHO_ANNUAL["no2"]} µg/m³)',  '#6B7280', ':'),
    (EU_AQD2030['no2'],  f'EU AQD 2030 ({EU_AQD2030["no2"]} µg/m³)', '#9CA3AF', '--'),
]:
    ax.axhline(limit, linestyle=ls, color=color, linewidth=1.5, alpha=0.7, label=label)

ax.set_xlabel('Year'); ax.set_ylabel('Annual mean NO₂ (µg/m³)')
ax.set_title('Portfolio NO₂ Trend 2018–2023\n'
             'Annotations show % change vs 2018 baseline', fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.2); ax.set_xticks(TREND_YEARS)
ax.set_xlim(TREND_YEARS[0] - 0.3, TREND_YEARS[-1] + 1.5)
plt.tight_layout()
plt.savefig('../exports/08_portfolio_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. CSRD Disclosure Table

Generate the ready-to-use disclosure table for your annual sustainability report.
This covers the ESRS E2 requirements for Pollution (AR 6 — material pollution at site level).

In [ ]:
# Combine annual means + exceedance into the CSRD disclosure format
csrd_rows = []
for i, site in enumerate(PORTFOLIO):
    s_row = df_screen[df_screen['Site'] == site['name']].iloc[0]
    e_row = df_exc[df_exc['Site'] == site['name']].iloc[0]

    no2 = s_row.get('NO2 µg/m³', None)
    pm25 = s_row.get('PM2P5 µg/m³', None)

    csrd_rows.append({
        'Reporting year':        YEAR,
        'Site name':             site['name'],
        'Employees':             site['employees'],
        'NO₂ annual mean (µg/m³)': no2,
        'PM2.5 annual mean (µg/m³)': pm25,
        'WHO NO₂ limit (µg/m³)': WHO_ANNUAL['no2'],
        'NO₂ vs WHO':            f"{no2/WHO_ANNUAL['no2']:.1f}×" if no2 else 'N/A',
        'EU AQD 2030 NO₂ (µg/m³)': EU_AQD2030['no2'],
        'EU AQD 2030 status':    '✅ Compliant' if (no2 or 999) <= EU_AQD2030['no2'] else '⚠️ Non-compliant',
        'Hourly WHO exceedance (h/yr)': e_row['Hours above WHO 25 µg/m³'],
        'Exceedance (% of year)': e_row['% of year'],
        'Data source':           'CAMS EAC4 reanalysis via Jiskta API',
        'Methodology':           'ESRS E2 / CAMS 0.1° grid, annual mean',
    })

df_csrd = pd.DataFrame(csrd_rows)

# Export to Excel
csrd_path = f'../exports/CSRD_AirQuality_{YEAR}.xlsx'
df_csrd.to_excel(csrd_path, index=False)
print(f'CSRD disclosure table saved to: {csrd_path}')
display(df_csrd[['Site name', 'NO₂ annual mean (µg/m³)', 'NO₂ vs WHO',
                  'EU AQD 2030 status', 'Hourly WHO exceedance (h/yr)']].set_index('Site name'))

## 5. Key Regulatory Reference

| Standard | Pollutant | Limit | Type | In force |
|----------|-----------|-------|------|----------|
| WHO 2021 | NO₂ | **10 µg/m³** annual | Health guideline | Now |
| WHO 2021 | NO₂ | **25 µg/m³** 1-hour | Health guideline | Now |
| WHO 2021 | PM2.5 | **5 µg/m³** annual | Health guideline | Now |
| EU AQD (current) | NO₂ | **40 µg/m³** annual | Legal limit | Until 2030 |
| EU AQD 2024 revision | NO₂ | **20 µg/m³** annual | Legal limit | 2030 |
| EU AQD 2024 revision | PM2.5 | **10 µg/m³** annual | Legal limit | 2030 |

### ESRS E2 Disclosure Requirements (Pollution)
- **Disclosure Requirement E2-4:** Amount of pollutants emitted to air, water, and soil
- **Application Requirement AR 6:** Include site-level concentrations for material locations
- **Relevant metrics:** Annual mean concentration, hours above WHO hourly guideline, trend vs baseline year

> **Data citation for reports:**  
> *"Air quality data sourced from CAMS EAC4 global reanalysis via Jiskta Climate API (jiskta.com).
>  Data resolution: 0.1° (~11 km grid). Concentrations represent area-averaged values and may be
>  lower than point measurements at street level by approximately 25–50%."*